# EDA

Open the CSV file in this directory.

In [ ]:
import pandas as pd

csv_path = "steam_reviews 1.csv"
df = pd.read_csv(csv_path)
df.head(10)

In [ ]:
df.info()

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
analyzer.lexicon["sick"] = 2.5

df["sentiment_score"] = df["review"].fillna("").apply(
    lambda review: analyzer.polarity_scores(str(review))["compound"]
)

df.to_csv("steam_reviews_with_sentiment.csv", index=False)
df.head(20)

In [ ]:
game_summary = df.groupby("title", as_index=False).agg(
    average_hour_played=("hour_played", "mean"),
    recommended_proportion=("recommendation", lambda x: (x == "Recommended").mean()),
    average_sentiment_score=("sentiment_score", "mean"),
    review_count=("review", "size"),
)

game_summary.to_csv("game_sentiment_summary.csv", index=False)
game_summary.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_cols = ["average_hour_played", "recommended_proportion", "average_sentiment_score"]

pairplot = sns.pairplot(game_summary[numeric_cols])
pairplot.fig.suptitle("Pairplot of Game Summary Numerical Variables", y=1.02)
pairplot.savefig("game_summary_pairplot.png", dpi=150, bbox_inches="tight")

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ["average_hour_played", "recommended_proportion", "average_sentiment_score", "review_count"]
scaler = StandardScaler()

game_summary_standardized = game_summary.copy()
game_summary_standardized[numeric_cols] = scaler.fit_transform(game_summary[numeric_cols])

game_summary_standardized.to_csv("game_sentiment_summary_standardized.csv", index=False)
game_summary_standardized.head()

In [ ]:
game_summary_standardized['average_sentiment_score'].mean()

In [ ]:
from sklearn.decomposition import PCA

numeric_cols = ["average_hour_played", "recommended_proportion", "average_sentiment_score", "review_count"]
pca = PCA(n_components=1)

game_pca_scores = game_summary_standardized[["title"]].copy()
game_pca_scores["pca_score"] = pca.fit_transform(game_summary_standardized[numeric_cols]).ravel()

game_pca_scores.to_csv("game_pca_scores.csv", index=False)
game_pca_scores.head()

In [ ]:
features = numeric_cols

pca_loadings = pd.Series(
    pca.components_[0],
    index=features
)

pca_loadings

In [ ]:
game_weighted_scores = game_summary_standardized[["title"]].copy()

game_weighted_scores["weighted_game_score"] = (
    0.5 * game_summary_standardized["average_hour_played"]
    + 0.25 * game_summary_standardized["recommended_proportion"]
    + 0.25 * game_summary_standardized["average_sentiment_score"]
    + 0.0 * game_summary_standardized["review_count"]
)

game_weighted_scores = game_weighted_scores.sort_values("weighted_game_score", ascending=False)
game_weighted_scores.to_csv("game_weighted_scores.csv", index=False)
game_weighted_scores.head(10)